Cellule 1 — Imports & config


In [ ]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

RAW_FILE  = Path("../data/raw/bq-results-last-12-months.csv")
CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(RAW_FILE, low_memory=False)
print(f"Lignes : {len(df):,}  |  Colonnes : {len(df.columns)}")
df.head(3)

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["GLOBALEVENTID"])
print(f"Doublons supprimés : {before - len(df):,}  →  {len(df):,} lignes")

In [ ]:
df["SQLDATE"] = pd.to_datetime(df["SQLDATE"].astype(str), format="%Y%m%d", errors="coerce")
df["year"]    = df["SQLDATE"].dt.year
df["month"]   = df["SQLDATE"].dt.month
df["week"]    = df["SQLDATE"].dt.isocalendar().week.astype("Int64")
print(f"Période : {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")

In [ ]:
for col, low, high in [("GoldsteinScale", -10.0, 10.0), ("AvgTone", None, None)]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    if low is not None: df[col] = df[col].clip(lower=low)
    if high is not None: df[col] = df[col].clip(upper=high)
    median = df[col].median()
    nans   = df[col].isna().sum()
    df[col] = df[col].fillna(median)
    print(f"{col:20s}  NaN imputés={nans}  médiane={median:.4f}")

In [ ]:
TEXT_COLS = ["Actor1Name","Actor2Name","Actor1CountryCode","Actor2CountryCode",
             "Actor1Type1Code","Actor2Type1Code","ActionGeo_FullName"]
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")
print("Nulls texte gérés ✓")

In [ ]:
df.to_csv(CLEAN_DIR / "gdelt_benin_clean.csv", index=False)
df.to_parquet(CLEAN_DIR / "gdelt_benin_clean.parquet", index=False, engine="pyarrow")
print("Sauvegardé dans data/clean/ ✓")